
# 01 — Make Raw Files Comparable

This is the second notebook in the final Economic Resilience project pipeline.

It takes the raw files produced by:

```text
00_Data_Acquisition_2002.ipynb
```

and converts them into a comparable country-year-variable structure.

## Final decisions reflected here

- Source coverage starts from **2002**.
- Input raw file names use **`2002_2025`**.
- No final country sample is selected here.
- No imputation is done here.
- WGI/governance is **not** part of the final POSet ordering set.
- Final POSet ordering variables are **5**, not 6.
- Public debt canonicalization is created here after country-code standardization.

## Final 5 ordering raw variables

These raw variables support the final 5 POSet ordering concepts:

1. `public_debt_gdp_canonical` → debt capacity
2. `unemployment_rate` → employment strength
3. `rd_gdp` → R&D intensity
4. `tertiary_education_25_64` → human capital
5. `energy_import_dependency_raw` → energy security

## Validation variables kept for later

The notebook also keeps variables used later for recovery and shock validation:

- `gdp_growth`
- `inflation_cpi`
- `productivity_gdp_per_hour`
- `public_debt_gdp_canonical`
- `unemployment_rate`

## Output logic

The main output is a standardized long table:

```text
Country | Year | variable | Value | source_file | source_route
```

Country codes are standardized to ISO3 wherever possible.


## v2 correction after first audit

The first audit revealed invalid standardized country codes such as `E` and `O`, created when aggregate codes like `EA21`, `EU27`, `EU19OECD`, `EU22OECD`, and `OECD_REP` passed through `country_converter`.

This v2 version fixes that by:

- adding those aggregate codes to the explicit aggregate-drop list;
- rejecting any country-converter output that is not a 3-letter ISO-style code;
- saving `invalid_standardized_country_codes.csv`;
- dropping invalid standardized codes before duplicate checks and downstream outputs.


In [1]:

# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import json
import re
import shutil
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 250)
pd.set_option("display.float_format", "{:.4f}".format)

try:
    import country_converter as coco
except Exception:
    coco = None
    print("country_converter not available. Using built-in fallback mappings where possible.")

PROJECT_ROOT = Path.cwd()

RAW_FILES_DIR = PROJECT_ROOT / "Data" / "Raw" / "Raw_Files"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Comparable_Raw_Files"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "01_Make_Raw_Files_Comparable"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

START_YEAR = 2002
END_YEAR = 2025

print("Run ID:", RUN_ID)
print("Run timestamp:", RUN_TIMESTAMP)
print("Project root:", PROJECT_ROOT.resolve())
print("Raw files folder:", RAW_FILES_DIR.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())
print("Diagnostics folder:", DIAGNOSTICS_DIR.resolve())


Run ID: 20260624_175307
Run timestamp: 2026-06-24 17:53:07
Project root: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24
Raw files folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Raw\Raw_Files
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Comparable_Raw_Files
Diagnostics folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Diagnostics\01_Make_Raw_Files_Comparable


In [2]:

# ------------------------------------------------------
# RAW FILE REGISTRY
# ------------------------------------------------------
# WGI is intentionally not processed here.
# It has a separate notebook because the WGI package structure differs from the simple Country-Year-Value CSVs.

RAW_FILE_REGISTRY = [
    {
        "dataset_label": "OECD_RD_GDP",
        "file_name": "OECD_RD_GDP_2002_2025.csv",
        "variable": "rd_gdp",
        "source_route": "OECD_REST",
        "country_code_type": "ISO3",
        "role": "final_ordering_input",
        "concept": "R&D expenditure as % of GDP",
        "higher_is_better_later": True,
    },
    {
        "dataset_label": "OECD_Inflation_CPI",
        "file_name": "OECD_Inflation_CPI_2002_2025.csv",
        "variable": "inflation_cpi",
        "source_route": "OECD_REST",
        "country_code_type": "ISO3",
        "role": "validation_or_sensitivity_input",
        "concept": "CPI inflation, annual growth rate",
        "higher_is_better_later": False,
    },
    {
        "dataset_label": "OECD_Unemployment_Rate",
        "file_name": "OECD_Unemployment_Rate_2002_2025.csv",
        "variable": "unemployment_rate",
        "source_route": "OECD_REST",
        "country_code_type": "ISO3",
        "role": "final_ordering_input_and_validation_input",
        "concept": "Unemployment rate, total sex and total age",
        "higher_is_better_later": False,
    },
    {
        "dataset_label": "OECD_Tertiary_Education",
        "file_name": "OECD_Tertiary_Education_2002_2025.csv",
        "variable": "tertiary_education_25_64",
        "source_route": "OECD_REST",
        "country_code_type": "ISO3",
        "role": "final_ordering_input",
        "concept": "Adult tertiary educational attainment, age 25–64",
        "higher_is_better_later": True,
    },
    {
        "dataset_label": "OECD_Productivity_GDP_per_Hour",
        "file_name": "OECD_Productivity_GDP_per_Hour_2002_2025.csv",
        "variable": "productivity_gdp_per_hour",
        "source_route": "DBNOMICS_OECD_MIRROR",
        "country_code_type": "ISO3",
        "role": "validation_or_sensitivity_input",
        "concept": "GDP per hour worked, constant prices",
        "higher_is_better_later": True,
    },
    {
        "dataset_label": "OECD_GDP_Growth",
        "file_name": "OECD_GDP_Growth_2002_2025.csv",
        "variable": "gdp_growth",
        "source_route": "DBNOMICS_OECD_MIRROR",
        "country_code_type": "ISO3",
        "role": "recovery_and_validation_input",
        "concept": "Real GDP growth rate",
        "higher_is_better_later": True,
    },
    {
        "dataset_label": "Eurostat_Public_Debt_GDP",
        "file_name": "Eurostat_Public_Debt_GDP_2002_2025.csv",
        "variable": "public_debt_gdp_eurostat",
        "source_route": "EUROSTAT_API",
        "country_code_type": "EUROSTAT_GEO",
        "role": "public_debt_source_input",
        "concept": "General government gross debt as % of GDP",
        "higher_is_better_later": False,
    },
    {
        "dataset_label": "WorldBank_Public_Debt_GDP",
        "file_name": "WorldBank_Public_Debt_GDP_2002_2025.csv",
        "variable": "public_debt_gdp_worldbank",
        "source_route": "WORLD_BANK_API",
        "country_code_type": "ISO3",
        "role": "public_debt_source_input",
        "concept": "Central government debt as % of GDP",
        "higher_is_better_later": False,
    },
    {
        "dataset_label": "WorldBank_Energy_Import_Dependency_Raw",
        "file_name": "WorldBank_Energy_Import_Dependency_Raw_2002_2025.csv",
        "variable": "energy_import_dependency_raw",
        "source_route": "WORLD_BANK_API",
        "country_code_type": "ISO3",
        "role": "final_ordering_input",
        "concept": "Energy imports, net % of energy use",
        "higher_is_better_later": False,
    },
]

approved_raw_file_registry = pd.DataFrame(RAW_FILE_REGISTRY)

approved_raw_file_registry.to_csv(PROCESSED_DIR / "approved_raw_file_registry.csv", index=False)
approved_raw_file_registry.to_csv(DIAGNOSTICS_DIR / "approved_raw_file_registry.csv", index=False)

display(approved_raw_file_registry)


,dataset_label,file_name,variable,source_route,country_code_type,role,concept,higher_is_better_later
0,OECD_RD_GDP,OECD_RD_GDP_2002_2025.csv,rd_gdp,OECD_REST,ISO3,final_ordering_input,R&D expenditure as % of GDP,True
1,OECD_Inflation_CPI,OECD_Inflation_CPI_2002_2025.csv,inflation_cpi,OECD_REST,ISO3,validation_or_sensitivity_input,"CPI inflation, annual growth rate",False
2,OECD_Unemployment_Rate,OECD_Unemployment_Rate_2002_2025.csv,unemployment_rate,OECD_REST,ISO3,final_ordering_input_and_validation_input,"Unemployment rate, total sex and total age",False
3,OECD_Tertiary_Education,OECD_Tertiary_Education_2002_2025.csv,tertiary_education_25_64,OECD_REST,ISO3,final_ordering_input,"Adult tertiary educational attainment, age 25–64",True
4,OECD_Productivity_GDP_per_Hour,OECD_Productivity_GDP_per_Hour_2002_2025.csv,productivity_gdp_per_hour,DBNOMICS_OECD_MIRROR,ISO3,validation_or_sensitivity_input,"GDP per hour worked, constant prices",True
5,OECD_GDP_Growth,OECD_GDP_Growth_2002_2025.csv,gdp_growth,DBNOMICS_OECD_MIRROR,ISO3,recovery_and_validation_input,Real GDP growth rate,True
6,Eurostat_Public_Debt_GDP,Eurostat_Public_Debt_GDP_2002_2025.csv,public_debt_gdp_eurostat,EUROSTAT_API,EUROSTAT_GEO,public_debt_source_input,General government gross debt as % of GDP,False
7,WorldBank_Public_Debt_GDP,WorldBank_Public_Debt_GDP_2002_2025.csv,public_debt_gdp_worldbank,WORLD_BANK_API,ISO3,public_debt_source_input,Central government debt as % of GDP,False
8,WorldBank_Energy_Import_Dependency_Raw,WorldBank_Energy_Import_Dependency_Raw_2002_20...,energy_import_dependency_raw,WORLD_BANK_API,ISO3,final_ordering_input,"Energy imports, net % of energy use",False


In [3]:

# ------------------------------------------------------
# FINAL VARIABLE GROUPS
# ------------------------------------------------------

FINAL_5_ORDERING_RAW_VARIABLES = [
    "public_debt_gdp_canonical",
    "unemployment_rate",
    "rd_gdp",
    "tertiary_education_25_64",
    "energy_import_dependency_raw",
]

VALIDATION_RAW_VARIABLES = [
    "gdp_growth",
    "unemployment_rate",
    "inflation_cpi",
    "public_debt_gdp_canonical",
    "productivity_gdp_per_hour",
]

SOURCE_PUBLIC_DEBT_VARIABLES = [
    "public_debt_gdp_eurostat",
    "public_debt_gdp_worldbank",
]

print("Final 5 ordering raw variables:")
for v in FINAL_5_ORDERING_RAW_VARIABLES:
    print("-", v)

print("\nValidation raw variables:")
for v in VALIDATION_RAW_VARIABLES:
    print("-", v)


Final 5 ordering raw variables:
- public_debt_gdp_canonical
- unemployment_rate
- rd_gdp
- tertiary_education_25_64
- energy_import_dependency_raw

Validation raw variables:
- gdp_growth
- unemployment_rate
- inflation_cpi
- public_debt_gdp_canonical
- productivity_gdp_per_hour


In [4]:

# ------------------------------------------------------
# COUNTRY CODE STANDARDIZATION HELPERS
# ------------------------------------------------------

# Common aggregate / non-country codes from OECD, Eurostat, World Bank, etc.
AGGREGATE_OR_REGION_CODES = {
    "OECD", "OECDE", "EU", "EU27_2020", "EU28", "EU27_2007", "EU15",
    "EA", "EA19", "EA20", "EA12",
    "G7M", "G7", "G20", "G20EA", "G20M", "WLD", "WORLD",
    "EA21", "EU19OECD", "EU22OECD", "EU25", "EU27", "OECD_REP",
    "HIC", "LIC", "LMC", "LMY", "MIC", "UMC", "OED", "EMU",
    "EAP", "EAS", "ECA", "ECS", "LCN", "MEA", "MNA", "NAC",
    "SAS", "SSA", "SSF", "TEA", "TEC", "TLA", "TMN", "TSA",
    "TSS", "CEB", "CSS", "PRE", "PST", "IBD", "IBT", "IDA",
    "IDX", "IDB", "LTE", "FCS", "AFE", "AFW",
}

# Eurostat special geo codes.
MANUAL_COUNTRY_CODE_MAP = {
    # Eurostat quirks
    "EL": "GRC",
    "GR": "GRC",
    "UK": "GBR",
    "XK": "XKX",

    # ISO2 fallback for Europe / common codes
    "AT": "AUT", "BE": "BEL", "BG": "BGR", "CH": "CHE", "CY": "CYP",
    "CZ": "CZE", "DE": "DEU", "DK": "DNK", "EE": "EST", "ES": "ESP",
    "FI": "FIN", "FR": "FRA", "HR": "HRV", "HU": "HUN", "IE": "IRL",
    "IS": "ISL", "IT": "ITA", "LI": "LIE", "LT": "LTU", "LU": "LUX",
    "LV": "LVA", "MT": "MLT", "NL": "NLD", "NO": "NOR", "PL": "POL",
    "PT": "PRT", "RO": "ROU", "SE": "SWE", "SI": "SVN", "SK": "SVK",
    "TR": "TUR",

    # ISO3 already but sometimes nonstandard
    "KOR": "KOR",
    "USA": "USA",
    "GBR": "GBR",
}


def standardize_country_code(code):
    original = code
    if pd.isna(code):
        return None, "missing"

    code = str(code).strip().upper()

    if code == "" or code in {"NAN", "NONE"}:
        return None, "missing"

    if code in AGGREGATE_OR_REGION_CODES:
        return None, "aggregate_or_region"

    if code in MANUAL_COUNTRY_CODE_MAP:
        mapped = MANUAL_COUNTRY_CODE_MAP[code]
        if mapped in {"XKX"}:
            return mapped, "manual_non_iso_or_special"
        return mapped, "manual_map"

    # If already ISO3-like, keep it.
    if re.fullmatch(r"[A-Z]{3}", code):
        return code, "kept_iso3_like"

    # Try country_converter for unusual cases.
    if coco is not None:
        try:
            converted = coco.convert(names=[code], to="ISO3", not_found=None)[0]
            if converted is not None and str(converted).lower() != "not found":
                converted = str(converted).upper().strip()

                # country_converter can sometimes map aggregate groups to invalid
                # one-letter pseudo-codes such as "E" or "O". These must not enter
                # the country-year panel.
                if converted in AGGREGATE_OR_REGION_CODES:
                    return None, "aggregate_or_region_after_conversion"

                if not re.fullmatch(r"[A-Z]{3}", converted):
                    return None, "invalid_country_converter_output"

                return converted, "country_converter"
        except Exception:
            pass

    return None, "unmapped"


def read_country_year_value_csv(path, registry_row):
    df = pd.read_csv(path)

    required_cols = {"Country", "Year", "Value"}
    missing_cols = required_cols - set(df.columns)

    if missing_cols:
        raise ValueError(f"{path.name}: missing required columns {missing_cols}")

    out = df[["Country", "Year", "Value"]].copy()
    out["source_country_code"] = out["Country"].astype(str).str.strip()
    out["Year"] = pd.to_numeric(out["Year"], errors="coerce")
    out["Value"] = pd.to_numeric(out["Value"], errors="coerce")

    mapped = out["source_country_code"].apply(standardize_country_code)
    out["Country"] = mapped.apply(lambda x: x[0])
    out["country_standardization_status"] = mapped.apply(lambda x: x[1])

    out["variable"] = registry_row["variable"]
    out["dataset_label"] = registry_row["dataset_label"]
    out["source_file"] = registry_row["file_name"]
    out["source_route"] = registry_row["source_route"]
    out["concept"] = registry_row["concept"]
    out["role"] = registry_row["role"]

    out = out.dropna(subset=["Year", "Value"]).copy()
    out["Year"] = out["Year"].astype(int)

    out = out[(out["Year"] >= START_YEAR) & (out["Year"] <= END_YEAR)].copy()

    return out


In [5]:

# ------------------------------------------------------
# READ AND STANDARDIZE RAW CSV FILES
# ------------------------------------------------------

standardized_frames = []
diagnostic_rows = []

for registry_row in RAW_FILE_REGISTRY:
    file_name = registry_row["file_name"]
    file_path = RAW_FILES_DIR / file_name

    print("=" * 90)
    print("Processing:", file_name)

    diagnostic = {
        "dataset_label": registry_row["dataset_label"],
        "file_name": file_name,
        "variable": registry_row["variable"],
        "source_route": registry_row["source_route"],
        "expected_path": str(file_path),
        "file_exists": file_path.exists(),
        "read_successfully": False,
        "rows_raw_after_read": np.nan,
        "rows_kept_after_standardization": np.nan,
        "countries_kept": np.nan,
        "min_year": np.nan,
        "max_year": np.nan,
        "error": np.nan,
    }

    if not file_path.exists():
        diagnostic["error"] = "File not found."
        diagnostic_rows.append(diagnostic)
        print("  -> Missing file.")
        continue

    try:
        temp = read_country_year_value_csv(file_path, registry_row)

        diagnostic["read_successfully"] = True
        diagnostic["rows_raw_after_read"] = len(pd.read_csv(file_path))
        diagnostic["rows_kept_after_standardization"] = len(temp[temp["Country"].notna()])
        diagnostic["countries_kept"] = temp["Country"].dropna().nunique()
        diagnostic["min_year"] = temp["Year"].min() if len(temp) else np.nan
        diagnostic["max_year"] = temp["Year"].max() if len(temp) else np.nan

        standardized_frames.append(temp)
        print(f"  -> Read and standardized: {len(temp):,} rows")

    except Exception as e:
        diagnostic["error"] = str(e)
        print("  -> ERROR:", e)

    diagnostic_rows.append(diagnostic)

csv_standardization_diagnostics = pd.DataFrame(diagnostic_rows)

if not standardized_frames:
    raise RuntimeError("No raw files could be standardized. Stop here.")

standardized_raw_long_pre_debt = pd.concat(standardized_frames, ignore_index=True)

csv_standardization_diagnostics.to_csv(DIAGNOSTICS_DIR / "csv_standardization_diagnostics.csv", index=False)
csv_standardization_diagnostics.to_csv(PROCESSED_DIR / "csv_standardization_diagnostics.csv", index=False)

display(csv_standardization_diagnostics)
display(standardized_raw_long_pre_debt.head())


Processing: OECD_RD_GDP_2002_2025.csv
  -> Read and standardized: 1,044 rows
Processing: OECD_Inflation_CPI_2002_2025.csv
  -> Read and standardized: 1,257 rows
Processing: OECD_Unemployment_Rate_2002_2025.csv
  -> Read and standardized: 1,338 rows
Processing: OECD_Tertiary_Education_2002_2025.csv
  -> Read and standardized: 943 rows
Processing: OECD_Productivity_GDP_per_Hour_2002_2025.csv
  -> Read and standardized: 991 rows
Processing: OECD_GDP_Growth_2002_2025.csv
  -> Read and standardized: 1,115 rows
Processing: Eurostat_Public_Debt_GDP_2002_2025.csv
  -> Read and standardized: 744 rows
Processing: WorldBank_Public_Debt_GDP_2002_2025.csv
  -> Read and standardized: 1,160 rows
Processing: WorldBank_Energy_Import_Dependency_Raw_2002_2025.csv
  -> Read and standardized: 3,798 rows


,dataset_label,file_name,variable,source_route,expected_path,file_exists,read_successfully,rows_raw_after_read,rows_kept_after_standardization,countries_kept,min_year,max_year,error
0,OECD_RD_GDP,OECD_RD_GDP_2002_2025.csv,rd_gdp,OECD_REST,D:\Milano Bicocca\Course Materials\1st Year\02...,True,True,1044,998,47,2002,2025,NaN
1,OECD_Inflation_CPI,OECD_Inflation_CPI_2002_2025.csv,inflation_cpi,OECD_REST,D:\Milano Bicocca\Course Materials\1st Year\02...,True,True,1257,1116,48,2002,2025,NaN
2,OECD_Unemployment_Rate,OECD_Unemployment_Rate_2002_2025.csv,unemployment_rate,OECD_REST,D:\Milano Bicocca\Course Materials\1st Year\02...,True,True,1338,1194,53,2002,2025,NaN
3,OECD_Tertiary_Education,OECD_Tertiary_Education_2002_2025.csv,tertiary_education_25_64,OECD_REST,D:\Milano Bicocca\Course Materials\1st Year\02...,True,True,943,915,48,2002,2024,NaN
4,OECD_Productivity_GDP_per_Hour,OECD_Productivity_GDP_per_Hour_2002_2025.csv,productivity_gdp_per_hour,DBNOMICS_OECD_MIRROR,D:\Milano Bicocca\Course Materials\1st Year\02...,True,True,991,923,41,2002,2024,NaN
5,OECD_GDP_Growth,OECD_GDP_Growth_2002_2025.csv,gdp_growth,DBNOMICS_OECD_MIRROR,D:\Milano Bicocca\Course Materials\1st Year\02...,True,True,1115,1043,44,2002,2025,NaN
6,Eurostat_Public_Debt_GDP,Eurostat_Public_Debt_GDP_2002_2025.csv,public_debt_gdp_eurostat,EUROSTAT_API,D:\Milano Bicocca\Course Materials\1st Year\02...,True,True,744,648,27,2002,2025,NaN
7,WorldBank_Public_Debt_GDP,WorldBank_Public_Debt_GDP_2002_2025.csv,public_debt_gdp_worldbank,WORLD_BANK_API,D:\Milano Bicocca\Course Materials\1st Year\02...,True,True,1160,1060,83,2002,2024,NaN
8,WorldBank_Energy_Import_Dependency_Raw,WorldBank_Energy_Import_Dependency_Raw_2002_20...,energy_import_dependency_raw,WORLD_BANK_API,D:\Milano Bicocca\Course Materials\1st Year\02...,True,True,3798,3038,153,2002,2023,NaN


,Country,Year,Value,source_country_code,country_standardization_status,variable,dataset_label,source_file,source_route,concept,role
0,ARG,2002,0.3478,ARG,kept_iso3_like,rd_gdp,OECD_RD_GDP,OECD_RD_GDP_2002_2025.csv,OECD_REST,R&D expenditure as % of GDP,final_ordering_input
1,ARG,2003,0.3668,ARG,kept_iso3_like,rd_gdp,OECD_RD_GDP,OECD_RD_GDP_2002_2025.csv,OECD_REST,R&D expenditure as % of GDP,final_ordering_input
2,ARG,2004,0.4038,ARG,kept_iso3_like,rd_gdp,OECD_RD_GDP,OECD_RD_GDP_2002_2025.csv,OECD_REST,R&D expenditure as % of GDP,final_ordering_input
3,ARG,2005,0.4207,ARG,kept_iso3_like,rd_gdp,OECD_RD_GDP,OECD_RD_GDP_2002_2025.csv,OECD_REST,R&D expenditure as % of GDP,final_ordering_input
4,ARG,2006,0.4522,ARG,kept_iso3_like,rd_gdp,OECD_RD_GDP,OECD_RD_GDP_2002_2025.csv,OECD_REST,R&D expenditure as % of GDP,final_ordering_input


In [6]:

# ------------------------------------------------------
# COUNTRY STANDARDIZATION DIAGNOSTICS
# ------------------------------------------------------

country_conversion_attempts = (
    standardized_raw_long_pre_debt
    .groupby(["source_country_code", "Country", "country_standardization_status"], dropna=False)
    .size()
    .reset_index(name="row_count")
    .sort_values(["country_standardization_status", "source_country_code"])
    .reset_index(drop=True)
)

unmapped_country_codes = country_conversion_attempts[
    country_conversion_attempts["country_standardization_status"].eq("unmapped")
].copy()

dropped_aggregate_regions = country_conversion_attempts[
    country_conversion_attempts["country_standardization_status"].str.contains("aggregate", na=False)
].copy()

country_standardization_status_summary = (
    standardized_raw_long_pre_debt
    .groupby(["dataset_label", "country_standardization_status"], dropna=False)
    .agg(
        rows=("Value", "size"),
        source_country_codes=("source_country_code", "nunique"),
        standardized_countries=("Country", "nunique"),
    )
    .reset_index()
    .sort_values(["dataset_label", "country_standardization_status"])
)

country_conversion_attempts.to_csv(DIAGNOSTICS_DIR / "country_conversion_attempts.csv", index=False)
country_standardization_status_summary.to_csv(DIAGNOSTICS_DIR / "country_standardization_status_summary.csv", index=False)
unmapped_country_codes.to_csv(DIAGNOSTICS_DIR / "unmapped_country_codes.csv", index=False)
dropped_aggregate_regions.to_csv(DIAGNOSTICS_DIR / "dropped_aggregate_regions.csv", index=False)

# Also save important diagnostics to processed folder for easy upload.
country_standardization_status_summary.to_csv(PROCESSED_DIR / "country_standardization_status_summary.csv", index=False)
unmapped_country_codes.to_csv(PROCESSED_DIR / "unmapped_country_codes.csv", index=False)
dropped_aggregate_regions.to_csv(PROCESSED_DIR / "dropped_aggregate_regions.csv", index=False)

print("Unmapped country-code rows:", len(unmapped_country_codes))
print("Aggregate/region rows:", len(dropped_aggregate_regions))

display(country_standardization_status_summary)
display(unmapped_country_codes.head(50))
display(dropped_aggregate_regions.head(50))


Unmapped country-code rows: 0
Aggregate/region rows: 50


,dataset_label,country_standardization_status,rows,source_country_codes,standardized_countries
0,Eurostat_Public_Debt_GDP,aggregate_or_region,96,4,0
1,Eurostat_Public_Debt_GDP,manual_map,648,27,27
2,OECD_GDP_Growth,aggregate_or_region,72,3,0
3,OECD_GDP_Growth,kept_iso3_like,971,41,41
4,OECD_GDP_Growth,manual_map,72,3,3
5,OECD_Inflation_CPI,aggregate_or_region,141,6,0
6,OECD_Inflation_CPI,kept_iso3_like,1045,45,45
7,OECD_Inflation_CPI,manual_map,71,3,3
8,OECD_Productivity_GDP_per_Hour,aggregate_or_region,68,3,0
9,OECD_Productivity_GDP_per_Hour,kept_iso3_like,865,38,38


,source_country_code,Country,country_standardization_status,row_count


,source_country_code,Country,country_standardization_status,row_count
0,AFE,NaN,aggregate_or_region,22
1,AFW,NaN,aggregate_or_region,21
2,CEB,NaN,aggregate_or_region,22
3,CSS,NaN,aggregate_or_region,23
4,EA,NaN,aggregate_or_region,24
5,EA19,NaN,aggregate_or_region,24
6,EA20,NaN,aggregate_or_region,71
7,EA21,NaN,aggregate_or_region,24
8,EAP,NaN,aggregate_or_region,21
9,EAS,NaN,aggregate_or_region,22


In [7]:

# ------------------------------------------------------
# DROP UNMAPPED / AGGREGATE ROWS
# ------------------------------------------------------

standardized_raw_long_pre_debt = standardized_raw_long_pre_debt[
    standardized_raw_long_pre_debt["Country"].notna()
].copy()

standardized_raw_long_pre_debt = standardized_raw_long_pre_debt[
    ~standardized_raw_long_pre_debt["country_standardization_status"].str.contains("aggregate", na=False)
].copy()

standardized_raw_long_pre_debt = (
    standardized_raw_long_pre_debt
    .sort_values(["variable", "Country", "Year", "source_route"])
    .reset_index(drop=True)
)

# Drop any invalid non-ISO3 outputs defensively.
invalid_standardized_country_codes = standardized_raw_long_pre_debt[
    ~standardized_raw_long_pre_debt["Country"].astype(str).str.match(r"^[A-Z]{3}$", na=False)
].copy()

invalid_standardized_country_codes.to_csv(
    DIAGNOSTICS_DIR / "invalid_standardized_country_codes.csv",
    index=False,
)

if len(invalid_standardized_country_codes) > 0:
    print("Invalid standardized country codes found and dropped:")
    display(
        invalid_standardized_country_codes[
            ["source_country_code", "Country", "country_standardization_status", "dataset_label"]
        ].drop_duplicates().sort_values(["dataset_label", "source_country_code"])
    )

standardized_raw_long_pre_debt = standardized_raw_long_pre_debt[
    standardized_raw_long_pre_debt["Country"].astype(str).str.match(r"^[A-Z]{3}$", na=False)
].copy()

print("Rows after dropping unmapped/aggregate/invalid rows:", len(standardized_raw_long_pre_debt))
print("Countries:", standardized_raw_long_pre_debt["Country"].nunique())
print("Variables:", standardized_raw_long_pre_debt["variable"].nunique())
display(standardized_raw_long_pre_debt.head())


Rows after dropping unmapped/aggregate/invalid rows: 10935
Countries: 177
Variables: 9


,Country,Year,Value,source_country_code,country_standardization_status,variable,dataset_label,source_file,source_route,concept,role
0,AGO,2002,-575.4543,AGO,kept_iso3_like,energy_import_dependency_raw,WorldBank_Energy_Import_Dependency_Raw,WorldBank_Energy_Import_Dependency_Raw_2002_20...,WORLD_BANK_API,"Energy imports, net % of energy use",final_ordering_input
1,AGO,2003,-520.1810,AGO,kept_iso3_like,energy_import_dependency_raw,WorldBank_Energy_Import_Dependency_Raw,WorldBank_Energy_Import_Dependency_Raw_2002_20...,WORLD_BANK_API,"Energy imports, net % of energy use",final_ordering_input
2,AGO,2004,-590.3533,AGO,kept_iso3_like,energy_import_dependency_raw,WorldBank_Energy_Import_Dependency_Raw,WorldBank_Energy_Import_Dependency_Raw_2002_20...,WORLD_BANK_API,"Energy imports, net % of energy use",final_ordering_input
3,AGO,2005,-812.8839,AGO,kept_iso3_like,energy_import_dependency_raw,WorldBank_Energy_Import_Dependency_Raw,WorldBank_Energy_Import_Dependency_Raw_2002_20...,WORLD_BANK_API,"Energy imports, net % of energy use",final_ordering_input
4,AGO,2006,-818.5898,AGO,kept_iso3_like,energy_import_dependency_raw,WorldBank_Energy_Import_Dependency_Raw,WorldBank_Energy_Import_Dependency_Raw_2002_20...,WORLD_BANK_API,"Energy imports, net % of energy use",final_ordering_input


In [8]:

# ------------------------------------------------------
# DUPLICATE CHECK BEFORE CANONICAL DEBT
# ------------------------------------------------------

duplicate_country_year_variable_check = (
    standardized_raw_long_pre_debt
    .groupby(["Country", "Year", "variable"])
    .size()
    .reset_index(name="duplicate_count")
    .query("duplicate_count > 1")
    .sort_values(["variable", "Country", "Year"])
    .reset_index(drop=True)
)

duplicate_country_year_variable_check.to_csv(DIAGNOSTICS_DIR / "duplicate_country_year_variable_check.csv", index=False)
duplicate_country_year_variable_check.to_csv(PROCESSED_DIR / "duplicate_country_year_variable_check.csv", index=False)

print("Duplicate Country-Year-Variable rows:", len(duplicate_country_year_variable_check))
display(duplicate_country_year_variable_check.head(50))

# If exact duplicates exist within the same variable, collapse by mean as a protective step.
# With the acquisition notebook this should usually not be needed.
standardized_raw_long_pre_debt = (
    standardized_raw_long_pre_debt
    .groupby([
        "Country", "Year", "variable",
        "dataset_label", "source_file", "source_route", "concept", "role"
    ], dropna=False)
    .agg(
        Value=("Value", "mean"),
        source_country_code=("source_country_code", "first"),
        country_standardization_status=("country_standardization_status", "first"),
    )
    .reset_index()
)


Duplicate Country-Year-Variable rows: 0


,Country,Year,variable,duplicate_count


In [9]:

# ------------------------------------------------------
# BUILD CANONICAL PUBLIC DEBT AFTER STANDARDIZATION
# ------------------------------------------------------
# Rule:
# 1. Prefer Eurostat public debt where available.
# 2. Use World Bank where Eurostat is missing.
#
# This is done after country-code standardization so both sources use Country ISO3.

debt_sources = standardized_raw_long_pre_debt[
    standardized_raw_long_pre_debt["variable"].isin(SOURCE_PUBLIC_DEBT_VARIABLES)
].copy()

if debt_sources.empty:
    raise ValueError("No public debt source rows found. Cannot build public_debt_gdp_canonical.")

source_priority = {
    "public_debt_gdp_eurostat": 1,
    "public_debt_gdp_worldbank": 2,
}

debt_sources["debt_source_priority"] = debt_sources["variable"].map(source_priority)

public_debt_source_comparison_country_year = (
    debt_sources
    .pivot_table(
        index=["Country", "Year"],
        columns="variable",
        values="Value",
        aggfunc="first",
    )
    .reset_index()
)

public_debt_source_comparison_country_year["has_eurostat"] = public_debt_source_comparison_country_year.get("public_debt_gdp_eurostat", pd.Series(index=public_debt_source_comparison_country_year.index, dtype=float)).notna()
public_debt_source_comparison_country_year["has_worldbank"] = public_debt_source_comparison_country_year.get("public_debt_gdp_worldbank", pd.Series(index=public_debt_source_comparison_country_year.index, dtype=float)).notna()

if "public_debt_gdp_eurostat" not in public_debt_source_comparison_country_year.columns:
    public_debt_source_comparison_country_year["public_debt_gdp_eurostat"] = np.nan

if "public_debt_gdp_worldbank" not in public_debt_source_comparison_country_year.columns:
    public_debt_source_comparison_country_year["public_debt_gdp_worldbank"] = np.nan

public_debt_source_comparison_country_year["public_debt_gdp_canonical"] = np.where(
    public_debt_source_comparison_country_year["public_debt_gdp_eurostat"].notna(),
    public_debt_source_comparison_country_year["public_debt_gdp_eurostat"],
    public_debt_source_comparison_country_year["public_debt_gdp_worldbank"],
)

public_debt_source_comparison_country_year["public_debt_canonical_source"] = np.where(
    public_debt_source_comparison_country_year["public_debt_gdp_eurostat"].notna(),
    "Eurostat",
    "World Bank",
)

canonical_debt_long = public_debt_source_comparison_country_year[
    ["Country", "Year", "public_debt_gdp_canonical", "public_debt_canonical_source"]
].dropna(subset=["public_debt_gdp_canonical"]).copy()

canonical_debt_long = canonical_debt_long.rename(columns={"public_debt_gdp_canonical": "Value"})
canonical_debt_long["variable"] = "public_debt_gdp_canonical"
canonical_debt_long["dataset_label"] = "Public_Debt_GDP_Canonical"
canonical_debt_long["source_file"] = "derived_after_standardization"
canonical_debt_long["source_route"] = "EUROSTAT_PREFERRED_THEN_WORLDBANK"
canonical_debt_long["concept"] = "Canonical public debt as % of GDP. Eurostat preferred where available; otherwise World Bank."
canonical_debt_long["role"] = "final_ordering_input_and_validation_input"
canonical_debt_long["source_country_code"] = canonical_debt_long["Country"]
canonical_debt_long["country_standardization_status"] = "derived_after_standardization"

# Keep the source for diagnostics as an extra column. Base long table can carry it too.
canonical_debt_long["public_debt_canonical_source"] = canonical_debt_long["public_debt_canonical_source"]

standardized_raw_long = pd.concat(
    [standardized_raw_long_pre_debt, canonical_debt_long],
    ignore_index=True,
    sort=False,
)

public_debt_source_comparison_country_year.to_csv(DIAGNOSTICS_DIR / "public_debt_source_comparison_country_year.csv", index=False)
canonical_debt_long.to_csv(PROCESSED_DIR / "public_debt_gdp_canonical_long.csv", index=False)

public_debt_source_comparison_overall_summary = pd.DataFrame([{
    "rows_with_eurostat": int(public_debt_source_comparison_country_year["has_eurostat"].sum()),
    "rows_with_worldbank": int(public_debt_source_comparison_country_year["has_worldbank"].sum()),
    "rows_with_both": int((public_debt_source_comparison_country_year["has_eurostat"] & public_debt_source_comparison_country_year["has_worldbank"]).sum()),
    "canonical_rows": int(canonical_debt_long.shape[0]),
    "countries": int(canonical_debt_long["Country"].nunique()),
    "min_year": int(canonical_debt_long["Year"].min()),
    "max_year": int(canonical_debt_long["Year"].max()),
}])

public_debt_source_comparison_summary_by_country = (
    canonical_debt_long
    .groupby(["Country", "public_debt_canonical_source"])
    .agg(rows=("Value", "size"), min_year=("Year", "min"), max_year=("Year", "max"))
    .reset_index()
)

public_debt_source_comparison_overall_summary.to_csv(DIAGNOSTICS_DIR / "public_debt_source_comparison_overall_summary.csv", index=False)
public_debt_source_comparison_summary_by_country.to_csv(DIAGNOSTICS_DIR / "public_debt_source_comparison_summary_by_country.csv", index=False)

display(public_debt_source_comparison_overall_summary)
display(public_debt_source_comparison_country_year.head())
display(canonical_debt_long.head())


,rows_with_eurostat,rows_with_worldbank,rows_with_both,canonical_rows,countries,min_year,max_year
0,648,1060,46,1662,108,2002,2025


variable,Country,Year,public_debt_gdp_eurostat,public_debt_gdp_worldbank,has_eurostat,has_worldbank,public_debt_gdp_canonical,public_debt_canonical_source
0,ALB,2011,NaN,69.1922,False,True,69.1922,World Bank
1,ALB,2012,NaN,64.0504,False,True,64.0504,World Bank
2,ALB,2013,NaN,70.4663,False,True,70.4663,World Bank
3,ALB,2014,NaN,72.9443,False,True,72.9443,World Bank
4,ALB,2015,NaN,79.2843,False,True,79.2843,World Bank


variable,Country,Year,Value,public_debt_canonical_source,variable,dataset_label,source_file,source_route,concept,role,source_country_code,country_standardization_status
0,ALB,2011,69.1922,World Bank,public_debt_gdp_canonical,Public_Debt_GDP_Canonical,derived_after_standardization,EUROSTAT_PREFERRED_THEN_WORLDBANK,Canonical public debt as % of GDP. Eurostat pr...,final_ordering_input_and_validation_input,ALB,derived_after_standardization
1,ALB,2012,64.0504,World Bank,public_debt_gdp_canonical,Public_Debt_GDP_Canonical,derived_after_standardization,EUROSTAT_PREFERRED_THEN_WORLDBANK,Canonical public debt as % of GDP. Eurostat pr...,final_ordering_input_and_validation_input,ALB,derived_after_standardization
2,ALB,2013,70.4663,World Bank,public_debt_gdp_canonical,Public_Debt_GDP_Canonical,derived_after_standardization,EUROSTAT_PREFERRED_THEN_WORLDBANK,Canonical public debt as % of GDP. Eurostat pr...,final_ordering_input_and_validation_input,ALB,derived_after_standardization
3,ALB,2014,72.9443,World Bank,public_debt_gdp_canonical,Public_Debt_GDP_Canonical,derived_after_standardization,EUROSTAT_PREFERRED_THEN_WORLDBANK,Canonical public debt as % of GDP. Eurostat pr...,final_ordering_input_and_validation_input,ALB,derived_after_standardization
4,ALB,2015,79.2843,World Bank,public_debt_gdp_canonical,Public_Debt_GDP_Canonical,derived_after_standardization,EUROSTAT_PREFERRED_THEN_WORLDBANK,Canonical public debt as % of GDP. Eurostat pr...,final_ordering_input_and_validation_input,ALB,derived_after_standardization


In [10]:

# ------------------------------------------------------
# SAVE STANDARDIZED LONG TABLE
# ------------------------------------------------------

# Recommended main output.
standardized_raw_long = (
    standardized_raw_long
    .sort_values(["variable", "Country", "Year"])
    .reset_index(drop=True)
)

standardized_raw_long.to_csv(PROCESSED_DIR / "standardized_country_year_variable_long.csv", index=False)
standardized_raw_long.to_csv(PROCESSED_DIR / "standardized_all_variables_long.csv", index=False)

# Compatibility copies under diagnostics.
standardized_raw_long.to_csv(DIAGNOSTICS_DIR / "standardized_country_year_variable_long.csv", index=False)

print("Standardized long rows:", len(standardized_raw_long))
print("Countries:", standardized_raw_long["Country"].nunique())
print("Variables:", standardized_raw_long["variable"].nunique())
display(standardized_raw_long.head())


Standardized long rows: 12597
Countries: 177
Variables: 10


,Country,Year,variable,dataset_label,source_file,source_route,concept,role,Value,source_country_code,country_standardization_status,public_debt_canonical_source
0,AGO,2002,energy_import_dependency_raw,WorldBank_Energy_Import_Dependency_Raw,WorldBank_Energy_Import_Dependency_Raw_2002_20...,WORLD_BANK_API,"Energy imports, net % of energy use",final_ordering_input,-575.4543,AGO,kept_iso3_like,NaN
1,AGO,2003,energy_import_dependency_raw,WorldBank_Energy_Import_Dependency_Raw,WorldBank_Energy_Import_Dependency_Raw_2002_20...,WORLD_BANK_API,"Energy imports, net % of energy use",final_ordering_input,-520.1810,AGO,kept_iso3_like,NaN
2,AGO,2004,energy_import_dependency_raw,WorldBank_Energy_Import_Dependency_Raw,WorldBank_Energy_Import_Dependency_Raw_2002_20...,WORLD_BANK_API,"Energy imports, net % of energy use",final_ordering_input,-590.3533,AGO,kept_iso3_like,NaN
3,AGO,2005,energy_import_dependency_raw,WorldBank_Energy_Import_Dependency_Raw,WorldBank_Energy_Import_Dependency_Raw_2002_20...,WORLD_BANK_API,"Energy imports, net % of energy use",final_ordering_input,-812.8839,AGO,kept_iso3_like,NaN
4,AGO,2006,energy_import_dependency_raw,WorldBank_Energy_Import_Dependency_Raw,WorldBank_Energy_Import_Dependency_Raw_2002_20...,WORLD_BANK_API,"Energy imports, net % of energy use",final_ordering_input,-818.5898,AGO,kept_iso3_like,NaN


In [11]:

# ------------------------------------------------------
# VARIABLE INVENTORY AND VALUE RANGE SANITY CHECKS
# ------------------------------------------------------

standardized_variable_inventory = (
    standardized_raw_long
    .groupby("variable")
    .agg(
        rows=("Value", "size"),
        countries=("Country", "nunique"),
        min_year=("Year", "min"),
        max_year=("Year", "max"),
        min_value=("Value", "min"),
        p01=("Value", lambda x: x.quantile(0.01)),
        median_value=("Value", "median"),
        p99=("Value", lambda x: x.quantile(0.99)),
        max_value=("Value", "max"),
        source_routes=("source_route", lambda x: ", ".join(sorted(set(x.astype(str))))),
        roles=("role", lambda x: ", ".join(sorted(set(x.astype(str))))),
    )
    .reset_index()
    .sort_values("variable")
)

def range_note(row):
    var = row["variable"]
    min_v = row["min_value"]
    max_v = row["max_value"]

    if var == "energy_import_dependency_raw":
        return "Raw energy dependency can be negative for net exporters and above 100 for high import dependence. Do not clip here."
    if "public_debt_gdp" in var and min_v < 0:
        return "Warning: negative public debt value detected. Check if it affects final analysis countries."
    if var == "unemployment_rate" and min_v < 0:
        return "Warning: unemployment rate below 0."
    if var == "tertiary_education_25_64" and (min_v < 0 or max_v > 100):
        return "Warning: tertiary attainment outside expected 0–100 range."
    if var == "rd_gdp" and min_v < 0:
        return "Warning: R&D/GDP below 0."
    return "OK"

value_range_sanity_checks = standardized_variable_inventory.copy()
value_range_sanity_checks["sanity_note"] = value_range_sanity_checks.apply(range_note, axis=1)

standardized_variable_inventory.to_csv(PROCESSED_DIR / "standardized_variable_inventory.csv", index=False)
standardized_variable_inventory.to_csv(DIAGNOSTICS_DIR / "standardized_variable_inventory.csv", index=False)

value_range_sanity_checks.to_csv(PROCESSED_DIR / "value_range_sanity_checks.csv", index=False)
value_range_sanity_checks.to_csv(DIAGNOSTICS_DIR / "value_range_sanity_checks.csv", index=False)

display(standardized_variable_inventory)
display(value_range_sanity_checks[["variable", "min_value", "max_value", "sanity_note"]])


,variable,rows,countries,min_year,max_year,min_value,p01,median_value,p99,max_value,source_routes,roles
0,energy_import_dependency_raw,3038,153,2002,2023,-2902.1927,-852.0874,22.7291,248.3580,450.1815,WORLD_BANK_API,final_ordering_input
1,gdp_growth,1043,44,2002,2025,-16.0400,-8.2724,2.5477,11.0368,24.6240,DBNOMICS_OECD_MIRROR,recovery_and_validation_input
2,inflation_cpi,1116,48,2002,2025,-4.4475,-0.9315,2.5000,34.7906,219.8845,OECD_REST,validation_or_sensitivity_input
3,productivity_gdp_per_hour,923,41,2002,2024,11.7448,14.2764,51.7788,114.8838,142.9044,DBNOMICS_OECD_MIRROR,validation_or_sensitivity_input
4,public_debt_gdp_canonical,1662,108,2002,2025,-1.1707,5.9922,50.2523,164.1780,209.4000,EUROSTAT_PREFERRED_THEN_WORLDBANK,final_ordering_input_and_validation_input
5,public_debt_gdp_eurostat,648,27,2002,2025,3.9000,6.4990,54.4000,181.3010,209.4000,EUROSTAT_API,public_debt_source_input
6,public_debt_gdp_worldbank,1060,83,2002,2024,-1.1707,6.1628,49.9645,152.6892,194.6828,WORLD_BANK_API,public_debt_source_input
7,rd_gdp,998,47,2002,2025,0.1639,0.2205,1.5462,4.8504,6.9589,OECD_REST,final_ordering_input
8,tertiary_education_25_64,915,48,2002,2024,5.4441,8.1334,31.3142,56.3198,64.6622,OECD_REST,final_ordering_input
9,unemployment_rate,1194,53,2002,2025,0.2510,0.9385,6.5585,29.3361,36.0250,OECD_REST,final_ordering_input_and_validation_input


,variable,min_value,max_value,sanity_note
0,energy_import_dependency_raw,-2902.1927,450.1815,Raw energy dependency can be negative for net ...
1,gdp_growth,-16.0400,24.6240,OK
2,inflation_cpi,-4.4475,219.8845,OK
3,productivity_gdp_per_hour,11.7448,142.9044,OK
4,public_debt_gdp_canonical,-1.1707,209.4000,Warning: negative public debt value detected. ...
5,public_debt_gdp_eurostat,3.9000,209.4000,OK
6,public_debt_gdp_worldbank,-1.1707,194.6828,Warning: negative public debt value detected. ...
7,rd_gdp,0.1639,6.9589,OK
8,tertiary_education_25_64,5.4441,64.6622,OK
9,unemployment_rate,0.2510,36.0250,OK


In [12]:

# ------------------------------------------------------
# COUNTRY-VARIABLE COVERAGE MATRICES
# ------------------------------------------------------

country_variable_presence_matrix = (
    standardized_raw_long
    .assign(present=1)
    .pivot_table(
        index="Country",
        columns="variable",
        values="present",
        aggfunc="max",
        fill_value=0,
    )
    .reset_index()
)

country_variable_year_count_matrix = (
    standardized_raw_long
    .pivot_table(
        index="Country",
        columns="variable",
        values="Year",
        aggfunc=lambda x: x.nunique(),
        fill_value=0,
    )
    .reset_index()
)

for var in FINAL_5_ORDERING_RAW_VARIABLES:
    if var not in country_variable_presence_matrix.columns:
        country_variable_presence_matrix[var] = 0
    if var not in country_variable_year_count_matrix.columns:
        country_variable_year_count_matrix[var] = 0

country_variable_presence_matrix["final5_ordering_variable_count_present"] = country_variable_presence_matrix[FINAL_5_ORDERING_RAW_VARIABLES].sum(axis=1)
country_variable_presence_matrix["complete_case_any_year_final5_ordering"] = country_variable_presence_matrix["final5_ordering_variable_count_present"].eq(len(FINAL_5_ORDERING_RAW_VARIABLES))

country_variable_year_count_matrix["final5_ordering_variable_total_years"] = country_variable_year_count_matrix[FINAL_5_ORDERING_RAW_VARIABLES].sum(axis=1)

country_variable_presence_matrix.to_csv(PROCESSED_DIR / "country_variable_presence_matrix.csv", index=False)
country_variable_presence_matrix.to_csv(DIAGNOSTICS_DIR / "country_variable_presence_matrix.csv", index=False)

country_variable_year_count_matrix.to_csv(PROCESSED_DIR / "country_variable_year_count_matrix.csv", index=False)
country_variable_year_count_matrix.to_csv(DIAGNOSTICS_DIR / "country_variable_year_count_matrix.csv", index=False)

display(country_variable_presence_matrix.head())
display(country_variable_year_count_matrix.head())


variable,Country,energy_import_dependency_raw,gdp_growth,inflation_cpi,productivity_gdp_per_hour,public_debt_gdp_canonical,public_debt_gdp_eurostat,public_debt_gdp_worldbank,rd_gdp,tertiary_education_25_64,unemployment_rate,final5_ordering_variable_count_present,complete_case_any_year_final5_ordering
0,AGO,1,0,0,0,0,0,0,0,0,0,1,False
1,ALB,1,0,0,0,1,0,1,0,0,0,2,False
2,AND,0,0,0,0,1,0,1,0,0,0,1,False
3,ARB,1,0,0,0,0,0,0,0,0,0,1,False
4,ARE,1,0,0,0,1,0,1,0,0,0,2,False


variable,Country,energy_import_dependency_raw,gdp_growth,inflation_cpi,productivity_gdp_per_hour,public_debt_gdp_canonical,public_debt_gdp_eurostat,public_debt_gdp_worldbank,rd_gdp,tertiary_education_25_64,unemployment_rate,final5_ordering_variable_total_years
0,AGO,21,0,0,0,0,0,0,0,0,0,21
1,ALB,22,0,0,0,14,0,14,0,0,0,36
2,AND,0,0,0,0,7,0,7,0,0,0,7
3,ARB,21,0,0,0,0,0,0,0,0,0,21
4,ARE,21,0,0,0,1,0,1,0,0,0,22


In [13]:

# ------------------------------------------------------
# FINAL 5 ORDERING VARIABLE BASELINE INPUT CHECK
# ------------------------------------------------------
# This is not final sample selection. It only checks whether the raw inputs exist
# in the two baseline years used for POSet construction.

BASELINE_YEARS = [2007, 2019]

baseline_input_rows = []

for baseline_year in BASELINE_YEARS:
    subset = standardized_raw_long[
        standardized_raw_long["Year"].eq(baseline_year)
        & standardized_raw_long["variable"].isin(FINAL_5_ORDERING_RAW_VARIABLES)
    ].copy()

    presence = (
        subset
        .assign(present=1)
        .pivot_table(
            index="Country",
            columns="variable",
            values="present",
            aggfunc="max",
            fill_value=0,
        )
    )

    for var in FINAL_5_ORDERING_RAW_VARIABLES:
        if var not in presence.columns:
            presence[var] = 0

    presence = presence[FINAL_5_ORDERING_RAW_VARIABLES]
    presence["baseline_year"] = baseline_year
    presence["final5_available_count"] = presence[FINAL_5_ORDERING_RAW_VARIABLES].sum(axis=1)
    presence["complete_case_final5_baseline_inputs"] = presence["final5_available_count"].eq(len(FINAL_5_ORDERING_RAW_VARIABLES))
    baseline_input_rows.append(presence.reset_index())

baseline_year_final5_input_check = pd.concat(baseline_input_rows, ignore_index=True)

baseline_year_final5_summary = (
    baseline_year_final5_input_check
    .groupby("baseline_year")
    .agg(
        countries_checked=("Country", "nunique"),
        complete_case_countries=("complete_case_final5_baseline_inputs", "sum"),
        mean_available_final5_variables=("final5_available_count", "mean"),
    )
    .reset_index()
)

baseline_year_final5_input_check.to_csv(PROCESSED_DIR / "baseline_year_final5_input_check.csv", index=False)
baseline_year_final5_input_check.to_csv(DIAGNOSTICS_DIR / "baseline_year_final5_input_check.csv", index=False)

baseline_year_final5_summary.to_csv(PROCESSED_DIR / "baseline_year_final5_summary.csv", index=False)
baseline_year_final5_summary.to_csv(DIAGNOSTICS_DIR / "baseline_year_final5_summary.csv", index=False)

display(baseline_year_final5_summary)
display(baseline_year_final5_input_check.head(80))


,baseline_year,countries_checked,complete_case_countries,mean_available_final5_variables
0,2007,152,25,2.1974
1,2019,161,35,2.2857


variable,Country,public_debt_gdp_canonical,unemployment_rate,rd_gdp,tertiary_education_25_64,energy_import_dependency_raw,baseline_year,final5_available_count,complete_case_final5_baseline_inputs
0,AGO,0,0,0,0,1,2007,1,False
1,ALB,0,0,0,0,1,2007,1,False
2,ARB,0,0,0,0,1,2007,1,False
3,ARE,0,0,0,0,1,2007,1,False
4,ARG,0,0,1,0,1,2007,2,False
5,ARM,0,0,0,0,1,2007,1,False
6,AUS,1,1,0,1,1,2007,4,False
7,AUT,1,1,1,1,1,2007,5,True
8,AZE,0,0,0,0,1,2007,1,False
9,BEL,1,1,1,1,1,2007,5,True


In [14]:

# ------------------------------------------------------
# VARIABLE METADATA
# ------------------------------------------------------

variable_metadata_rows = []

for row in RAW_FILE_REGISTRY:
    variable_metadata_rows.append({
        "variable": row["variable"],
        "dataset_label": row["dataset_label"],
        "source_file": row["file_name"],
        "source_route": row["source_route"],
        "concept": row["concept"],
        "role": row["role"],
        "higher_is_better_later": row["higher_is_better_later"],
        "is_final_ordering_variable": row["variable"] in FINAL_5_ORDERING_RAW_VARIABLES,
        "is_validation_variable": row["variable"] in VALIDATION_RAW_VARIABLES,
        "note": "Source variable from raw acquisition.",
    })

variable_metadata_rows.append({
    "variable": "public_debt_gdp_canonical",
    "dataset_label": "Public_Debt_GDP_Canonical",
    "source_file": "derived_after_standardization",
    "source_route": "EUROSTAT_PREFERRED_THEN_WORLDBANK",
    "concept": "Canonical public debt as % of GDP",
    "role": "final_ordering_input_and_validation_input",
    "higher_is_better_later": False,
    "is_final_ordering_variable": True,
    "is_validation_variable": True,
    "note": "Created after country-code standardization. Eurostat preferred where available; otherwise World Bank.",
})

variable_metadata_rows.append({
    "variable": "wgi_governance_contextual",
    "dataset_label": "WGI_Governance_Raw",
    "source_file": "WGI_CSV.zip",
    "source_route": "WORLD_BANK_DATABANK_BULK_DOWNLOAD",
    "concept": "Worldwide Governance Indicators",
    "role": "contextual_or_sensitivity_only",
    "higher_is_better_later": True,
    "is_final_ordering_variable": False,
    "is_validation_variable": False,
    "note": "Not processed in this notebook and not used as a final POSet ordering variable.",
})

variable_metadata = pd.DataFrame(variable_metadata_rows)

variable_metadata.to_csv(PROCESSED_DIR / "variable_metadata.csv", index=False)
variable_metadata.to_csv(DIAGNOSTICS_DIR / "variable_metadata.csv", index=False)

display(variable_metadata)


,variable,dataset_label,source_file,source_route,concept,role,higher_is_better_later,is_final_ordering_variable,is_validation_variable,note
0,rd_gdp,OECD_RD_GDP,OECD_RD_GDP_2002_2025.csv,OECD_REST,R&D expenditure as % of GDP,final_ordering_input,True,True,False,Source variable from raw acquisition.
1,inflation_cpi,OECD_Inflation_CPI,OECD_Inflation_CPI_2002_2025.csv,OECD_REST,"CPI inflation, annual growth rate",validation_or_sensitivity_input,False,False,True,Source variable from raw acquisition.
2,unemployment_rate,OECD_Unemployment_Rate,OECD_Unemployment_Rate_2002_2025.csv,OECD_REST,"Unemployment rate, total sex and total age",final_ordering_input_and_validation_input,False,True,True,Source variable from raw acquisition.
3,tertiary_education_25_64,OECD_Tertiary_Education,OECD_Tertiary_Education_2002_2025.csv,OECD_REST,"Adult tertiary educational attainment, age 25–64",final_ordering_input,True,True,False,Source variable from raw acquisition.
4,productivity_gdp_per_hour,OECD_Productivity_GDP_per_Hour,OECD_Productivity_GDP_per_Hour_2002_2025.csv,DBNOMICS_OECD_MIRROR,"GDP per hour worked, constant prices",validation_or_sensitivity_input,True,False,True,Source variable from raw acquisition.
5,gdp_growth,OECD_GDP_Growth,OECD_GDP_Growth_2002_2025.csv,DBNOMICS_OECD_MIRROR,Real GDP growth rate,recovery_and_validation_input,True,False,True,Source variable from raw acquisition.
6,public_debt_gdp_eurostat,Eurostat_Public_Debt_GDP,Eurostat_Public_Debt_GDP_2002_2025.csv,EUROSTAT_API,General government gross debt as % of GDP,public_debt_source_input,False,False,False,Source variable from raw acquisition.
7,public_debt_gdp_worldbank,WorldBank_Public_Debt_GDP,WorldBank_Public_Debt_GDP_2002_2025.csv,WORLD_BANK_API,Central government debt as % of GDP,public_debt_source_input,False,False,False,Source variable from raw acquisition.
8,energy_import_dependency_raw,WorldBank_Energy_Import_Dependency_Raw,WorldBank_Energy_Import_Dependency_Raw_2002_20...,WORLD_BANK_API,"Energy imports, net % of energy use",final_ordering_input,False,True,False,Source variable from raw acquisition.
9,public_debt_gdp_canonical,Public_Debt_GDP_Canonical,derived_after_standardization,EUROSTAT_PREFERRED_THEN_WORLDBANK,Canonical public debt as % of GDP,final_ordering_input_and_validation_input,False,True,True,Created after country-code standardization. Eu...


In [15]:

# ------------------------------------------------------
# AUDIT WORKBOOK
# ------------------------------------------------------

audit_xlsx = AUDIT_DIR / "01_Make_Raw_Files_Comparable_Audit.xlsx"

with pd.ExcelWriter(audit_xlsx, engine="xlsxwriter") as writer:
    approved_raw_file_registry.to_excel(writer, sheet_name="approved_raw_registry", index=False)
    csv_standardization_diagnostics.to_excel(writer, sheet_name="standardization_diag", index=False)
    country_standardization_status_summary.to_excel(writer, sheet_name="country_status_summary", index=False)
    country_conversion_attempts.to_excel(writer, sheet_name="country_conversion", index=False)
    unmapped_country_codes.to_excel(writer, sheet_name="unmapped_codes", index=False)
    dropped_aggregate_regions.to_excel(writer, sheet_name="dropped_aggregates", index=False)
    invalid_standardized_country_codes.to_excel(writer, sheet_name="invalid_std_codes", index=False)
    duplicate_country_year_variable_check.to_excel(writer, sheet_name="duplicates", index=False)
    standardized_variable_inventory.to_excel(writer, sheet_name="variable_inventory", index=False)
    value_range_sanity_checks.to_excel(writer, sheet_name="value_ranges", index=False)
    country_variable_presence_matrix.to_excel(writer, sheet_name="presence_matrix", index=False)
    country_variable_year_count_matrix.to_excel(writer, sheet_name="year_count_matrix", index=False)
    baseline_year_final5_summary.to_excel(writer, sheet_name="final5_baseline_summary", index=False)
    baseline_year_final5_input_check.to_excel(writer, sheet_name="final5_baseline_check", index=False)
    public_debt_source_comparison_overall_summary.to_excel(writer, sheet_name="debt_overall", index=False)
    public_debt_source_comparison_country_year.to_excel(writer, sheet_name="debt_country_year", index=False)
    variable_metadata.to_excel(writer, sheet_name="variable_metadata", index=False)

print("Audit workbook saved:", audit_xlsx.resolve())


Audit workbook saved: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Audit\01_Make_Raw_Files_Comparable_Audit.xlsx


In [16]:

# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

expected_outputs = [
    PROCESSED_DIR / "standardized_country_year_variable_long.csv",
    PROCESSED_DIR / "standardized_all_variables_long.csv",
    PROCESSED_DIR / "approved_raw_file_registry.csv",
    PROCESSED_DIR / "csv_standardization_diagnostics.csv",
    PROCESSED_DIR / "standardized_variable_inventory.csv",
    PROCESSED_DIR / "value_range_sanity_checks.csv",
    PROCESSED_DIR / "country_variable_presence_matrix.csv",
    PROCESSED_DIR / "country_variable_year_count_matrix.csv",
    PROCESSED_DIR / "baseline_year_final5_input_check.csv",
    PROCESSED_DIR / "baseline_year_final5_summary.csv",
    PROCESSED_DIR / "variable_metadata.csv",
    AUDIT_DIR / "01_Make_Raw_Files_Comparable_Audit.xlsx",
]

output_check = pd.DataFrame([
    {
        "output_path": str(path),
        "exists": path.exists(),
        "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else np.nan,
    }
    for path in expected_outputs
])

output_check.to_csv(DIAGNOSTICS_DIR / "output_check.csv", index=False)

print("01 MAKE RAW FILES COMPARABLE COMPLETE")
print("=" * 90)

display(output_check)

print("\nKey methodological checks:")
print("- Final POSet ordering variables: 5")
print("- WGI/governance included as context/sensitivity only: yes")
print("- Canonical public debt built after standardization: yes")
print("- Main standardized output: standardized_country_year_variable_long.csv")
print("- Next notebook: 02_Raw_Files_Coverage_Diagnostics_2002_5Var.ipynb")


01 MAKE RAW FILES COMPARABLE COMPLETE


,output_path,exists,size_kb
0,D:\Milano Bicocca\Course Materials\1st Year\02...,True,2754.0200
1,D:\Milano Bicocca\Course Materials\1st Year\02...,True,2754.0200
2,D:\Milano Bicocca\Course Materials\1st Year\02...,True,1.6200
3,D:\Milano Bicocca\Course Materials\1st Year\02...,True,2.8800
4,D:\Milano Bicocca\Course Materials\1st Year\02...,True,1.6400
5,D:\Milano Bicocca\Course Materials\1st Year\02...,True,1.9600
6,D:\Milano Bicocca\Course Materials\1st Year\02...,True,5.9600
7,D:\Milano Bicocca\Course Materials\1st Year\02...,True,5.6700
8,D:\Milano Bicocca\Course Materials\1st Year\02...,True,8.6800
9,D:\Milano Bicocca\Course Materials\1st Year\02...,True,0.1500



Key methodological checks:
- Final POSet ordering variables: 5
- WGI/governance included as context/sensitivity only: yes
- Canonical public debt built after standardization: yes
- Main standardized output: standardized_country_year_variable_long.csv
- Next notebook: 02_Raw_Files_Coverage_Diagnostics_2002_5Var.ipynb
